# Sequential feature selection with DataFrame pipelines

`SequentialFeatureSelector(..., preserve_dataframe=True)` keeps pandas column names, dtypes, and row indexes when evaluating feature subsets. This allows scikit-learn pipelines that use `make_column_selector` or estimators that use sample indexes to participate in feature selection.

The default, `preserve_dataframe=False`, retains the existing conversion to NumPy arrays. With the option enabled, `transform` and `fit_transform` also return DataFrames for DataFrame inputs. NumPy and sparse inputs retain their existing behavior.

This small synthetic example illustrates a categorical predictor alongside an unrelated numeric predictor. Its scores demonstrate the API behavior; they are not a benchmark of real-world prediction quality.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mlxtend.feature_selection import SequentialFeatureSelector

X = pd.DataFrame(
    {
        "amount": np.arange(80, dtype=float),
        "kind": pd.Categorical(["low", "high"] * 40),
    },
    index=np.arange(100, 180),
)
y = np.array([0, 1] * 40)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=1
)

model = make_pipeline(
    ColumnTransformer(
        [
            ("numeric", StandardScaler(), make_column_selector(dtype_include=np.number)),
            (
                "category",
                OneHotEncoder(handle_unknown="ignore"),
                make_column_selector(dtype_include="category"),
            ),
        ]
    ),
    LogisticRegression(),
)


The dtype selectors are evaluated on each candidate subset. A fixed list of column names in a `ColumnTransformer` must only refer to columns present in that subset; use callable column selectors when columns can be removed.

Keep preprocessing inside the estimator pipeline so it is fitted separately in each cross-validation training fold. Set `n_jobs=2` to evaluate candidate subsets in parallel if desired.


In [2]:
selector = SequentialFeatureSelector(
    model,
    k_features=1,
    scoring="accuracy",
    cv=3,
    preserve_dataframe=True,
)
selected_train = selector.fit_transform(X_train, y_train)
selected_test = selector.transform(X_test)

print("Selected features:", selector.k_feature_names_)
print("CV accuracy:", selector.k_score_)
print("Output type:", type(selected_train).__name__)
print("Column dtype:", selected_train["kind"].dtype)
print("Row index preserved:", selected_train.index.equals(X_train.index))

model.fit(selected_train, y_train)
print("Held-out accuracy:", model.score(selected_test, y_test))


Selected features: ('kind',)
CV accuracy: 1.0
Output type: DataFrame
Column dtype: category
Row index preserved: True
Held-out accuracy: 1.0


Feature subsets are selected by column position, including during `transform`. Pass columns in the same order used during fitting. Named `fixed_features` and `feature_groups` continue to work with distinct column names.

See [SequentialFeatureSelector](SequentialFeatureSelector.md) for forward, backward, floating, grouped, and fixed-feature selection options. This option applies to `SequentialFeatureSelector`; it does not change `ExhaustiveFeatureSelector`.
